# Datamart Analítico TechStore — ETL en Python
### Notebook `01_datamart_etl.ipynb`

Este notebook implementa el pipeline ETL (Extract, Transform, Load) del datamart de ventas de
**TechStore**, siguiendo un esquema estrella orientado a Power BI.

**Grano del hecho:** una línea de venta (un producto dentro de una boleta/orden).

**Dimensiones:** Cliente, Producto, Tienda, Tiempo y Promoción.

**Métricas (hechos):** `cantidad`, `precio_unitario`, `descuento`, `importe`, `costo_total`, `margen`.

**Estructura de carpetas esperada** (este notebook vive en `notebooks/`, al mismo nivel que `data/`):

```
proyecto/
├── data/
│   ├── raw/          <- CSV crudos (fact_ventas, dim_cliente, dim_producto, dim_tienda, dim_promocion, dim_tiempo)
│   └── processed/    <- se genera automáticamente al ejecutar este notebook
└── notebooks/
    └── 01_datamart_etl.ipynb   <- este archivo
```

**Problemas de calidad detectados en los datos crudos** que este pipeline corrige:

| Problema | Tabla / Columna afectada | Acción ETL |
|---|---|---|
| Fechas en 3 formatos distintos (ISO, DD/MM/YYYY, texto en español) | `fecha` en fact_ventas, `fecha_nacimiento`/`fecha_alta` en dim_cliente, `fecha_inicio`/`fecha_fin` en dim_promocion | Parseo flexible fila a fila |
| Registros duplicados por llave de negocio | fact_ventas (`id_venta` + `id_producto`) | `drop_duplicates` |
| Categorías de producto con mayúsculas/minúsculas y espacios inconsistentes | `categoria` en dim_producto | strip + upper + title |
| Valores nulos | `distrito` (dim_cliente), `categoria` (dim_producto), `descuento`/`id_promocion` (fact_ventas) | Imputación con reglas de negocio |
| Claves foráneas huérfanas | `id_producto` en fact_ventas que no existe en dim_producto | Eliminación de filas huérfanas |
| Descuento expresado en escala de puntos porcentuales (0–40) en vez de fracción (0–1) | `descuento` en fact_ventas | Se divide entre 100 al calcular `importe` |


## 1. Configuración del entorno

In [ ]:
# Celda 1 - Instalación de dependencias (descomentar si hace falta)
# !pip install pandas numpy


In [1]:
# Celda 2 - Importaciones y configuración de rutas
import pandas as pd
import numpy as np
import re
import os
import time
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

# Rutas relativas: este notebook vive en notebooks/, al mismo nivel que data/
RAW_DIR = '../data/raw'
PROC_DIR = '../data/processed'
os.makedirs(PROC_DIR, exist_ok=True)

inicio_pipeline = time.time()
print('Entorno listo:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))


Entorno listo: 2026-07-12 22:35:25


## 2. Extracción (E)

Se cargan las seis tablas crudas desde `data/raw`. Los archivos usan `;` como separador
(formato exportado desde Excel/Power BI en configuración regional latinoamericana).

In [2]:
# Celda 3 - Carga de datos crudos
fact_ventas_raw   = pd.read_csv(f'{RAW_DIR}/fact_ventas.csv',   sep=';', encoding='utf-8')
dim_cliente_raw   = pd.read_csv(f'{RAW_DIR}/dim_cliente.csv',   sep=';', encoding='utf-8')
dim_producto_raw  = pd.read_csv(f'{RAW_DIR}/dim_producto.csv',  sep=';', encoding='utf-8')
dim_tienda_raw    = pd.read_csv(f'{RAW_DIR}/dim_tienda.csv',    sep=';', encoding='utf-8')
dim_promocion_raw = pd.read_csv(f'{RAW_DIR}/dim_promocion.csv', sep=';', encoding='utf-8')
dim_tiempo_raw    = pd.read_csv(f'{RAW_DIR}/dim_tiempo.csv',    sep=';', encoding='utf-8')

print('=== REGISTROS EXTRAIDOS ===')
for nombre, df in [('fact_ventas', fact_ventas_raw), ('dim_cliente', dim_cliente_raw),
                    ('dim_producto', dim_producto_raw), ('dim_tienda', dim_tienda_raw),
                    ('dim_promocion', dim_promocion_raw), ('dim_tiempo', dim_tiempo_raw)]:
    print(f'  {nombre:<14}: {len(df):>7,} filas | {len(df.columns)} columnas')


=== REGISTROS EXTRAIDOS ===
  fact_ventas   :  60,900 filas | 9 columnas
  dim_cliente   :   5,000 filas | 7 columnas
  dim_producto  :     500 filas | 7 columnas
  dim_tienda    :      15 filas | 5 columnas
  dim_promocion :      40 filas | 6 columnas
  dim_tiempo    :     730 filas | 7 columnas


In [ ]:
# Celda 4 - Vista previa de la tabla de hechos cruda
fact_ventas_raw.head(5)


## 3. Métricas de calidad iniciales (línea base)

Se mide, para cada tabla: valores nulos, filas duplicadas y llaves foráneas huérfanas
(referencias en `fact_ventas` que no existen en la dimensión correspondiente). Estas
cifras son la línea base contra la que se comparará la calidad final del pipeline.

In [3]:
# Celda 5 - Funciones de medición de calidad
def calidad_tabla(df, nombre):
    """Reporta nulos y filas duplicadas de una tabla."""
    total = len(df)
    nulos = int(df.isnull().sum().sum())
    dups = int(df.duplicated().sum())
    print(f'  {nombre:<14}: filas={total:>7,} | nulos={nulos:>6,} | filas_duplicadas={dups}')
    return {'tabla': nombre, 'filas': total, 'nulos': nulos, 'duplicados': dups}


def contar_huerfanos(fact, fk, dim, pk, etiqueta):
    """Cuenta filas de fact cuya llave foránea fk no existe en dim[pk] (nulos se ignoran)."""
    validos = fact[fk].isin(dim[pk]) | fact[fk].isnull()
    huerfanos = int((~validos).sum())
    print(f'  Huerfanos en {etiqueta:<14}: {huerfanos}')
    return huerfanos


print('=== CALIDAD INICIAL (datos crudos) ===')
cal_inicial = {}
cal_inicial['fact_ventas']   = calidad_tabla(fact_ventas_raw, 'fact_ventas')
cal_inicial['dim_cliente']   = calidad_tabla(dim_cliente_raw, 'dim_cliente')
cal_inicial['dim_producto']  = calidad_tabla(dim_producto_raw, 'dim_producto')
cal_inicial['dim_tienda']    = calidad_tabla(dim_tienda_raw, 'dim_tienda')
cal_inicial['dim_promocion'] = calidad_tabla(dim_promocion_raw, 'dim_promocion')
cal_inicial['dim_tiempo']    = calidad_tabla(dim_tiempo_raw, 'dim_tiempo')


=== CALIDAD INICIAL (datos crudos) ===
  fact_ventas   : filas= 60,900 | nulos=35,928 | filas_duplicadas=291
  dim_cliente   : filas=  5,000 | nulos=   350 | filas_duplicadas=0
  dim_producto  : filas=    500 | nulos=    30 | filas_duplicadas=0
  dim_tienda    : filas=     15 | nulos=     0 | filas_duplicadas=0
  dim_promocion : filas=     40 | nulos=     0 | filas_duplicadas=0
  dim_tiempo    : filas=    730 | nulos=     0 | filas_duplicadas=0


In [4]:
# Celda 6 - Claves huérfanas y duplicados por llave de negocio (crudo)
print('=== CLAVES HUERFANAS EN fact_ventas (crudo) ===')
huerfanos_ini = {
    'id_producto':   contar_huerfanos(fact_ventas_raw, 'id_producto', dim_producto_raw, 'id_producto', 'id_producto'),
    'id_cliente':    contar_huerfanos(fact_ventas_raw, 'id_cliente', dim_cliente_raw, 'id_cliente', 'id_cliente'),
    'id_tienda':     contar_huerfanos(fact_ventas_raw, 'id_tienda', dim_tienda_raw, 'id_tienda', 'id_tienda'),
    'id_promocion':  contar_huerfanos(fact_ventas_raw, 'id_promocion', dim_promocion_raw, 'id_promocion', 'id_promocion'),
}

dup_negocio_ini = int(fact_ventas_raw.duplicated(subset=['id_venta', 'id_producto']).sum())
print(f'\nDuplicados por llave de negocio (id_venta + id_producto): {dup_negocio_ini}')


=== CLAVES HUERFANAS EN fact_ventas (crudo) ===
  Huerfanos en id_producto   : 307
  Huerfanos en id_cliente    : 0
  Huerfanos en id_tienda     : 0
  Huerfanos en id_promocion  : 0

Duplicados por llave de negocio (id_venta + id_producto): 900


## 4. Transformación (T)

### 4.1 Parseo flexible de fechas

El dataset mezcla **tres formatos de fecha** dentro de la misma columna: ISO (`YYYY-MM-DD`),
latino (`DD/MM/YYYY`) y texto en español (`D de mes de YYYY`). `pd.to_datetime` infiere un
único formato para toda la serie, por lo que con formatos mixtos falla silenciosamente y
convierte en `NaT` la mayoría de las fechas. Por eso se define un parser que evalúa cada
valor fila por fila contra los tres patrones posibles.

In [5]:
# Celda 7 - Parser flexible de fechas (ISO / DD-MM-AAAA / texto en español)
MESES_ES = {
    'enero': 1, 'febrero': 2, 'marzo': 3, 'abril': 4, 'mayo': 5, 'junio': 6,
    'julio': 7, 'agosto': 8, 'septiembre': 9, 'setiembre': 9, 'octubre': 10,
    'noviembre': 11, 'diciembre': 12,
}

PATRON_ISO = re.compile(r'^(\d{4})-(\d{1,2})-(\d{1,2})$')
PATRON_DMY = re.compile(r'^(\d{1,2})/(\d{1,2})/(\d{4})$')
PATRON_FECHA_ES = re.compile(r'^(\d{1,2})\s+de\s+([a-záéíóúñ]+)\s+de\s+(\d{4})$')


def _parsear_una_fecha(texto):
    if not isinstance(texto, str):
        return pd.NaT
    t = texto.strip()

    m = PATRON_ISO.match(t)
    if m:
        anio, mes, dia = m.groups()
        try:
            return pd.Timestamp(year=int(anio), month=int(mes), day=int(dia))
        except ValueError:
            return pd.NaT

    m = PATRON_DMY.match(t)
    if m:
        dia, mes, anio = m.groups()
        try:
            return pd.Timestamp(year=int(anio), month=int(mes), day=int(dia))
        except ValueError:
            return pd.NaT

    m = PATRON_FECHA_ES.match(t.lower())
    if m:
        dia, mes_nombre, anio = m.groups()
        mes = MESES_ES.get(mes_nombre)
        if mes is None:
            return pd.NaT
        try:
            return pd.Timestamp(year=int(anio), month=mes, day=int(dia))
        except ValueError:
            return pd.NaT

    return pd.NaT


def parsear_fecha_flexible(serie):
    """Convierte una serie de texto con fechas en 3 formatos distintos a datetime64."""
    return serie.apply(_parsear_una_fecha)


# Prueba rápida sobre fact_ventas
_prueba = parsear_fecha_flexible(fact_ventas_raw['fecha'])
print(f'Fechas sin poder parsear en fact_ventas.fecha: {_prueba.isnull().sum()} de {len(_prueba)}')


Fechas sin poder parsear en fact_ventas.fecha: 0 de 60900


### 4.2 Dimensión Cliente

- Se parsean `fecha_nacimiento` y `fecha_alta` con el parser flexible.
- `distrito` nulo se imputa con `'Desconocido'`.
- Se estandarizan textos (`nombre`, `sexo`, `segmento_programa`) con `strip` + `title`/`upper`.
- Se eliminan duplicados por `id_cliente`.

In [6]:
# Celda 8 - Transformación dim_cliente
dim_cliente = dim_cliente_raw.copy()

dim_cliente['fecha_nacimiento'] = parsear_fecha_flexible(dim_cliente['fecha_nacimiento'])
dim_cliente['fecha_alta'] = parsear_fecha_flexible(dim_cliente['fecha_alta'])

nulos_distrito_antes = int(dim_cliente['distrito'].isnull().sum())
dim_cliente['distrito'] = dim_cliente['distrito'].fillna('Desconocido').str.strip().str.title()

dim_cliente['nombre'] = dim_cliente['nombre'].str.strip().str.title()
dim_cliente['sexo'] = dim_cliente['sexo'].str.strip().str.upper()
dim_cliente['segmento_programa'] = dim_cliente['segmento_programa'].str.strip().str.title()

antes = len(dim_cliente)
dim_cliente = dim_cliente.drop_duplicates(subset=['id_cliente'])

print(f'distrito - nulos imputados como Desconocido: {nulos_distrito_antes}')
print(f'duplicados por id_cliente eliminados: {antes - len(dim_cliente)}')
print(f'fechas de nacimiento sin parsear: {dim_cliente["fecha_nacimiento"].isnull().sum()}')
print(f'fechas de alta sin parsear: {dim_cliente["fecha_alta"].isnull().sum()}')
print(f'dim_cliente final: {len(dim_cliente):,} filas | nulos totales: {dim_cliente.isnull().sum().sum()}')


distrito - nulos imputados como Desconocido: 350
duplicados por id_cliente eliminados: 0
fechas de nacimiento sin parsear: 0
fechas de alta sin parsear: 0
dim_cliente final: 5,000 filas | nulos totales: 0


### 4.3 Dimensión Producto

- `categoria` tiene espacios y capitalización inconsistente (`'  celulares  '`, `'CELULARES'`,
  `'Celulares'`, etc.) y 30 valores nulos. Se normaliza con `strip().upper()` (con lo que
  colapsan a 14 categorías canónicas) y se presenta en `title case`; los nulos se
  imputan como `'Sin Categoria'`.
- Se eliminan duplicados por `id_producto` y filas con precio o costo no positivos.

In [7]:
# Celda 9 - Transformación dim_producto
dim_producto = dim_producto_raw.copy()

nulos_categoria_antes = int(dim_producto['categoria'].isnull().sum())
dim_producto['categoria'] = dim_producto['categoria'].str.strip().str.upper()
dim_producto['categoria'] = dim_producto['categoria'].fillna('SIN CATEGORIA').str.title()

dim_producto['subcategoria'] = dim_producto['subcategoria'].str.strip().str.title()
dim_producto['marca'] = dim_producto['marca'].str.strip()
dim_producto['nombre'] = dim_producto['nombre'].str.strip()

antes = len(dim_producto)
dim_producto = dim_producto.drop_duplicates(subset=['id_producto'])
dim_producto = dim_producto[(dim_producto['precio_lista'] > 0) & (dim_producto['costo'] > 0)]

print(f'categoria - nulos imputados como Sin Categoria: {nulos_categoria_antes}')
print(f'filas eliminadas (duplicadas o precio/costo invalido): {antes - len(dim_producto)}')
print(f'categorias canonicas: {sorted(dim_producto["categoria"].unique())}')
print(f'dim_producto final: {len(dim_producto):,} filas | nulos totales: {dim_producto.isnull().sum().sum()}')


categoria - nulos imputados como Sin Categoria: 30
filas eliminadas (duplicadas o precio/costo invalido): 0
categorias canonicas: ['Accesorios', 'Almacenamiento', 'Audio', 'Celulares', 'Componentes', 'Computadoras', 'Gaming', 'Hogar Inteligente', 'Impresion', 'Laptops', 'Monitores', 'Perifericos', 'Redes', 'Sin Categoria', 'Tablets']
dim_producto final: 500 filas | nulos totales: 0


### 4.4 Dimensión Tienda

Tabla pequeña y ya limpia; se estandariza texto por consistencia y se eliminan duplicados
por `id_tienda`.

In [8]:
# Celda 10 - Transformación dim_tienda
dim_tienda = dim_tienda_raw.copy()
for col in ['nombre', 'canal', 'region', 'ciudad']:
    dim_tienda[col] = dim_tienda[col].str.strip().str.title()

dim_tienda = dim_tienda.drop_duplicates(subset=['id_tienda'])
print(f'dim_tienda final: {len(dim_tienda):,} filas | nulos totales: {dim_tienda.isnull().sum().sum()}')


dim_tienda final: 15 filas | nulos totales: 0


### 4.5 Dimensión Promoción

- Se parsean `fecha_inicio` y `fecha_fin` con el parser flexible.
- Se estandariza `tipo` con `title case`.
- Se agrega un **miembro "Sin Promoción" (`id_promocion = 0`)**: en `fact_ventas` unas
  32,883 filas no tienen promoción asociada (`id_promocion` nulo). Siguiendo la práctica de
  Kimball de "unknown member", se crea este registro para que esas ventas puedan tener una
  llave foránea válida en vez de perderse o quedar en `NULL`.

In [9]:
# Celda 11 - Transformación dim_promocion
dim_promocion = dim_promocion_raw.copy()

dim_promocion['nombre'] = dim_promocion['nombre'].str.strip()
dim_promocion['tipo'] = dim_promocion['tipo'].str.strip().str.title()
dim_promocion['fecha_inicio'] = parsear_fecha_flexible(dim_promocion['fecha_inicio'])
dim_promocion['fecha_fin'] = parsear_fecha_flexible(dim_promocion['fecha_fin'])

dim_promocion = dim_promocion.drop_duplicates(subset=['id_promocion'])

sin_promocion = pd.DataFrame([{
    'id_promocion': 0,
    'nombre': 'Sin Promocion',
    'tipo': 'Ninguna',
    'descuento_pct': 0,
    'fecha_inicio': pd.NaT,
    'fecha_fin': pd.NaT,
}])
dim_promocion = pd.concat([sin_promocion, dim_promocion], ignore_index=True)

print(f'fechas de inicio sin parsear: {dim_promocion["fecha_inicio"].isnull().sum() - 1}')
print(f'fechas de fin sin parsear: {dim_promocion["fecha_fin"].isnull().sum() - 1}')
print(f'dim_promocion final: {len(dim_promocion):,} filas (incluye miembro Sin Promocion)')


fechas de inicio sin parsear: 0
fechas de fin sin parsear: 0
dim_promocion final: 41 filas (incluye miembro Sin Promocion)


### 4.6 Dimensión Tiempo

Tabla ya bien formada (fechas ISO, sin nulos); solo se convierte `fecha` a `datetime64` y se
estandariza `dia_semana`.

In [10]:
# Celda 12 - Transformación dim_tiempo
dim_tiempo = dim_tiempo_raw.copy()
dim_tiempo['fecha'] = pd.to_datetime(dim_tiempo['fecha'], format='%Y-%m-%d', errors='coerce')
dim_tiempo['dia_semana'] = dim_tiempo['dia_semana'].str.strip().str.title()
dim_tiempo = dim_tiempo.drop_duplicates(subset=['fecha'])

print(f'fechas invalidas: {dim_tiempo["fecha"].isnull().sum()}')
print(f'dim_tiempo final: {len(dim_tiempo):,} filas')


fechas invalidas: 0
dim_tiempo final: 730 filas


### 4.7 Tabla de hechos `fact_ventas`

Pasos, en orden:

1. **Duplicados por llave de negocio** (`id_venta` + `id_producto`, el grano del hecho).
2. **Fechas**: parseo flexible; filas cuya fecha no pudo parsearse se descartan (no debería
   quedar ninguna, dado que el parser cubre los 3 formatos presentes).
3. **Nulos de negocio**: `id_promocion` nulo → `0` (miembro "Sin Promoción");
   `descuento` nulo → `0.0` (sin descuento).
4. **Integridad referencial**: se eliminan filas cuyo `id_producto` no exista en
   `dim_producto` (claves huérfanas — no se pueden imputar sin inventar datos).
5. **Rangos válidos**: `cantidad > 0`, `precio_unitario > 0`, `descuento` entre 0 y 100
   (el descuento está en puntos porcentuales, no en fracción).
6. Verificación final de integridad contra `id_cliente` e `id_tienda` (por completitud;
   en este dataset no se detectaron huérfanas en estas columnas).

In [11]:
# Celda 13 - Limpieza de fact_ventas
fact = fact_ventas_raw.copy()

# Paso 1: duplicados por llave de negocio
antes = len(fact)
fact = fact.drop_duplicates(subset=['id_venta', 'id_producto'])
print(f'Paso 1 - duplicados eliminados: {antes - len(fact)} | quedan {len(fact):,}')

# Paso 2: fechas
fact['fecha'] = parsear_fecha_flexible(fact['fecha'])
invalidas = int(fact['fecha'].isnull().sum())
fact = fact.dropna(subset=['fecha'])
print(f'Paso 2 - fechas invalidas eliminadas: {invalidas} | quedan {len(fact):,}')

# Paso 3: nulos de negocio
nulos_promo_antes = int(fact['id_promocion'].isnull().sum())
nulos_desc_antes = int(fact['descuento'].isnull().sum())
fact['id_promocion'] = fact['id_promocion'].fillna(0).astype(int)
fact['descuento'] = fact['descuento'].fillna(0.0)
print(f'Paso 3 - id_promocion nulos imputados (Sin Promocion): {nulos_promo_antes}')
print(f'         descuento nulos imputados (0.0): {nulos_desc_antes}')

# Paso 4: integridad referencial - producto
antes = len(fact)
fact = fact[fact['id_producto'].isin(dim_producto['id_producto'])]
print(f'Paso 4 - huerfanos id_producto eliminados: {antes - len(fact)} | quedan {len(fact):,}')

# Paso 5: rangos validos
antes = len(fact)
fact = fact[fact['cantidad'] > 0]
fact = fact[fact['precio_unitario'] > 0]
fact = fact[fact['descuento'].between(0, 100)]
print(f'Paso 5 - registros fuera de rango eliminados: {antes - len(fact)} | quedan {len(fact):,}')

# Paso 6: integridad referencial - cliente y tienda (defensivo)
antes = len(fact)
fact = fact[fact['id_cliente'].isin(dim_cliente['id_cliente'])]
fact = fact[fact['id_tienda'].isin(dim_tienda['id_tienda'])]
fact = fact[fact['id_promocion'].isin(dim_promocion['id_promocion'])]
print(f'Paso 6 - huerfanos cliente/tienda/promocion eliminados: {antes - len(fact)} | quedan {len(fact):,}')

print(f'\nNulos restantes en fact_ventas: {fact.isnull().sum().sum()}')


Paso 1 - duplicados eliminados: 900 | quedan 60,000
Paso 2 - fechas invalidas eliminadas: 0 | quedan 60,000
Paso 3 - id_promocion nulos imputados (Sin Promocion): 32375
         descuento nulos imputados (0.0): 2989
Paso 4 - huerfanos id_producto eliminados: 300 | quedan 59,700
Paso 5 - registros fuera de rango eliminados: 0 | quedan 59,700
Paso 6 - huerfanos cliente/tienda/promocion eliminados: 0 | quedan 59,700

Nulos restantes en fact_ventas: 0


### 4.8 Métricas derivadas

- `importe = cantidad × precio_unitario × (1 − descuento / 100)`
  (el descuento está en puntos porcentuales enteros, p. ej. `20` = 20%, por eso se divide entre 100).
- `costo_total = cantidad × costo` (el costo unitario viene de `dim_producto`).
- `margen = importe − costo_total`.

In [12]:
# Celda 14 - Métricas derivadas (importe, costo_total, margen)
fact = fact.merge(dim_producto[['id_producto', 'costo']], on='id_producto', how='left')

fact['importe'] = (fact['cantidad'] * fact['precio_unitario'] * (1 - fact['descuento'] / 100)).round(2)
fact['costo_total'] = (fact['cantidad'] * fact['costo']).round(2)
fact['margen'] = (fact['importe'] - fact['costo_total']).round(2)

fact_ventas = fact.drop(columns=['costo'])

print('Nuevas columnas:', [c for c in fact_ventas.columns if c not in fact_ventas_raw.columns])
print(f'fact_ventas final: {len(fact_ventas):,} filas | {len(fact_ventas.columns)} columnas')
fact_ventas.head(5)


Nuevas columnas: ['importe', 'costo_total', 'margen']
fact_ventas final: 59,700 filas | 12 columnas


,id_venta,fecha,id_cliente,id_producto,id_tienda,id_promocion,cantidad,precio_unitario,descuento,importe,costo_total,margen
0,17504,2025-10-12,2235,253,13,38,3,143.04,20.0,343.30,274.02,69.28
1,46116,2025-12-25,4031,426,4,0,1,346.19,0.0,346.19,242.97,103.22
2,59784,2024-08-27,4363,194,8,0,2,475.39,0.0,950.78,553.02,397.76
3,11462,2024-12-19,2934,426,10,5,2,313.40,30.0,438.76,485.94,-47.18
4,50030,2024-05-11,1351,46,3,0,1,536.19,0.0,536.19,366.20,169.99


## 5. Validación de integridad referencial final

Se confirma que, tras la limpieza, no quedan llaves foráneas huérfanas en `fact_ventas`
frente a ninguna de las cuatro dimensiones.

**Punto de control:**
- 0 huérfanos en `id_producto`, `id_cliente`, `id_tienda`, `id_promocion`.
- `fact_ventas` no debe tener valores nulos.
- Todas las dimensiones deben tener llave primaria única.

In [13]:
# Celda 15 - Verificación de integridad referencial
print('=== INTEGRIDAD REFERENCIAL FINAL ===')
huerfanos_final = {
    'id_producto':  contar_huerfanos(fact_ventas, 'id_producto', dim_producto, 'id_producto', 'id_producto'),
    'id_cliente':   contar_huerfanos(fact_ventas, 'id_cliente', dim_cliente, 'id_cliente', 'id_cliente'),
    'id_tienda':    contar_huerfanos(fact_ventas, 'id_tienda', dim_tienda, 'id_tienda', 'id_tienda'),
    'id_promocion': contar_huerfanos(fact_ventas, 'id_promocion', dim_promocion, 'id_promocion', 'id_promocion'),
}

assert all(v == 0 for v in huerfanos_final.values()), 'Hay llaves foraneas huerfanas sin resolver'
assert fact_ventas.isnull().sum().sum() == 0, 'fact_ventas tiene valores nulos'
assert dim_cliente['id_cliente'].is_unique, 'id_cliente no es unico en dim_cliente'
assert dim_producto['id_producto'].is_unique, 'id_producto no es unico en dim_producto'
assert dim_tienda['id_tienda'].is_unique, 'id_tienda no es unico en dim_tienda'
assert dim_promocion['id_promocion'].is_unique, 'id_promocion no es unico en dim_promocion'
assert dim_tiempo['fecha'].is_unique, 'fecha no es unica en dim_tiempo'

print('\nTodas las validaciones de integridad pasaron correctamente.')


=== INTEGRIDAD REFERENCIAL FINAL ===
  Huerfanos en id_producto   : 0
  Huerfanos en id_cliente    : 0
  Huerfanos en id_tienda     : 0
  Huerfanos en id_promocion  : 0

Todas las validaciones de integridad pasaron correctamente.


## 6. Carga (L): exportar tablas limpias a `data/processed`

In [14]:
# Celda 16 - Exportación a data/processed
def cargar_tabla(df, nombre, carpeta):
    ruta = f'{carpeta}/{nombre}.csv'
    df.to_csv(ruta, index=False, encoding='utf-8')
    print(f'  {nombre:<14} -> {len(df):>7,} filas  |  {ruta}')

print('Exportando tablas del modelo estrella...')
cargar_tabla(dim_cliente,   'dim_cliente',   PROC_DIR)
cargar_tabla(dim_producto,  'dim_producto',  PROC_DIR)
cargar_tabla(dim_tienda,    'dim_tienda',    PROC_DIR)
cargar_tabla(dim_promocion, 'dim_promocion', PROC_DIR)
cargar_tabla(dim_tiempo,    'dim_tiempo',    PROC_DIR)
cargar_tabla(fact_ventas,   'fact_ventas',   PROC_DIR)

fin_pipeline = time.time()
tiempo_total = round(fin_pipeline - inicio_pipeline, 2)
print(f'\nPipeline completado en {tiempo_total} segundos.')


Exportando tablas del modelo estrella...
  dim_cliente    ->   5,000 filas  |  ../data/processed/dim_cliente.csv
  dim_producto   ->     500 filas  |  ../data/processed/dim_producto.csv
  dim_tienda     ->      15 filas  |  ../data/processed/dim_tienda.csv
  dim_promocion  ->      41 filas  |  ../data/processed/dim_promocion.csv
  dim_tiempo     ->     730 filas  |  ../data/processed/dim_tiempo.csv
  fact_ventas    ->  59,700 filas  |  ../data/processed/fact_ventas.csv

Pipeline completado en 219.91 segundos.


## 7. Diccionario de datos

Se genera automáticamente a partir de las tablas ya limpias (campo, tipo, tabla y una
descripción de negocio) y se exporta a `data/processed/diccionario_datos.csv`, listo para
adjuntar a la documentación del datamart.

In [15]:
# Celda 17 - Generación del diccionario de datos
DESCRIPCIONES = {
    # fact_ventas
    'id_venta': 'Identificador de la boleta/orden de venta',
    'fecha': 'Fecha en que se realizo la venta (llave hacia dim_tiempo)',
    'id_cliente': 'Llave foranea hacia dim_cliente',
    'id_producto': 'Llave foranea hacia dim_producto',
    'id_tienda': 'Llave foranea hacia dim_tienda',
    'id_promocion': 'Llave foranea hacia dim_promocion (0 = sin promocion)',
    'cantidad': 'Unidades vendidas del producto en la linea de venta',
    'precio_unitario': 'Precio de venta por unidad, en soles',
    'descuento': 'Descuento aplicado, en puntos porcentuales (0-100)',
    'importe': 'cantidad x precio_unitario x (1 - descuento/100)',
    'costo_total': 'cantidad x costo unitario del producto',
    'margen': 'importe - costo_total',
    # dim_cliente
    'nombre': 'Nombre completo (persona o producto, segun tabla)',
    'sexo': 'Sexo del cliente (M/F)',
    'fecha_nacimiento': 'Fecha de nacimiento del cliente',
    'distrito': 'Distrito de residencia del cliente (Desconocido si no se registro)',
    'fecha_alta': 'Fecha de inscripcion del cliente al programa de fidelizacion',
    'segmento_programa': 'Nivel del programa de fidelizacion (Bronce/Plata/Oro/Platino)',
    # dim_producto
    'categoria': 'Categoria del producto, normalizada',
    'subcategoria': 'Subcategoria del producto',
    'marca': 'Marca del producto',
    'precio_lista': 'Precio de lista sugerido, en soles',
    'costo': 'Costo unitario del producto, en soles',
    # dim_tienda
    'canal': 'Canal de venta de la tienda (fisico/online)',
    'region': 'Region geografica de la tienda',
    'ciudad': 'Ciudad donde opera la tienda',
    # dim_promocion
    'tipo': 'Tipo de promocion (Bundle, Descuento %, Liquidacion, Cupon, 2x1, Ninguna)',
    'descuento_pct': 'Descuento nominal de la promocion, en puntos porcentuales',
    'fecha_inicio': 'Fecha de inicio de vigencia de la promocion',
    'fecha_fin': 'Fecha de fin de vigencia de la promocion',
    # dim_tiempo
    'dia': 'Dia del mes',
    'mes': 'Mes del anio (1-12)',
    'trimestre': 'Trimestre del anio (1-4)',
    'anio': 'Anio calendario',
    'dia_semana': 'Nombre del dia de la semana',
    'es_feriado': 'Indicador de feriado (1 = si, 0 = no)',
}

TABLAS = {
    'fact_ventas': fact_ventas,
    'dim_cliente': dim_cliente,
    'dim_producto': dim_producto,
    'dim_tienda': dim_tienda,
    'dim_promocion': dim_promocion,
    'dim_tiempo': dim_tiempo,
}

LLAVES = {
    'fact_ventas': {'id_venta', 'id_producto'},
    'dim_cliente': {'id_cliente'},
    'dim_producto': {'id_producto'},
    'dim_tienda': {'id_tienda'},
    'dim_promocion': {'id_promocion'},
    'dim_tiempo': {'fecha'},
}

filas_diccionario = []
for nombre_tabla, df in TABLAS.items():
    for col in df.columns:
        filas_diccionario.append({
            'tabla': nombre_tabla,
            'campo': col,
            'tipo_dato': str(df[col].dtype),
            'es_llave': col in LLAVES.get(nombre_tabla, set()),
            'descripcion': DESCRIPCIONES.get(col, ''),
        })

diccionario_datos = pd.DataFrame(filas_diccionario)
diccionario_datos.to_csv(f'{PROC_DIR}/diccionario_datos.csv', index=False, encoding='utf-8')
print(f'Diccionario de datos generado: {len(diccionario_datos)} campos -> {PROC_DIR}/diccionario_datos.csv')
diccionario_datos.head(10)


Diccionario de datos generado: 44 campos -> ../data/processed/diccionario_datos.csv


,tabla,campo,tipo_dato,es_llave,descripcion
0,fact_ventas,id_venta,int64,True,Identificador de la boleta/orden de venta
1,fact_ventas,fecha,datetime64[ns],False,Fecha en que se realizo la venta (llave hacia ...
2,fact_ventas,id_cliente,int64,False,Llave foranea hacia dim_cliente
3,fact_ventas,id_producto,int64,True,Llave foranea hacia dim_producto
4,fact_ventas,id_tienda,int64,False,Llave foranea hacia dim_tienda
5,fact_ventas,id_promocion,int64,False,Llave foranea hacia dim_promocion (0 = sin pro...
6,fact_ventas,cantidad,int64,False,Unidades vendidas del producto en la linea de ...
7,fact_ventas,precio_unitario,float64,False,"Precio de venta por unidad, en soles"
8,fact_ventas,descuento,float64,False,"Descuento aplicado, en puntos porcentuales (0-..."
9,fact_ventas,importe,float64,False,cantidad x precio_unitario x (1 - descuento/100)


## 8. Reporte de auditoría ETL

Se mide la calidad de `fact_ventas` ya transformada y se compara contra la línea base
inicial, dejando constancia en `data/processed/reporte_auditoria_etl.txt`.

In [16]:
# Celda 18 - Calidad final y reporte de auditoria
print('=== CALIDAD FINAL (fact_ventas transformada) ===')
cal_final_fact = calidad_tabla(fact_ventas, 'fact_ventas')
huerfanos_reporte = huerfanos_final

def generar_reporte(cal_ini, cal_fin, huerf_ini, huerf_fin, dup_negocio_ini, tiempo_seg, ruta):
    total_ini = cal_ini['fact_ventas']['filas']
    total_fin = cal_fin['filas']
    lineas = [
        '=' * 64,
        'REPORTE DE AUDITORIA - ETL DATAMART TECHSTORE',
        f'Fecha de ejecucion: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}',
        '=' * 64,
        '',
        '--- FUENTE (fact_ventas) ---',
        f'Registros extraidos          : {total_ini:,}',
        f'Duplicados por llave negocio  : {dup_negocio_ini}',
        f'Huerfanos id_producto (inicial): {huerf_ini["id_producto"]}',
        '',
        '--- TRANSFORMACION ---',
        f'Registros tras limpieza       : {total_fin:,}',
        f'Registros eliminados          : {total_ini - total_fin:,}',
        '',
        '--- CALIDAD DE DATOS (fact_ventas) ---',
        f'Nulos antes                   : {cal_ini["fact_ventas"]["nulos"]:,}',
        f'Nulos despues                 : {cal_fin["nulos"]:,}',
        f'Filas duplicadas antes        : {cal_ini["fact_ventas"]["duplicados"]}',
        f'Filas duplicadas despues      : {cal_fin["duplicados"]}',
        '',
        '--- INTEGRIDAD REFERENCIAL FINAL ---',
        f'Huerfanos id_producto          : {huerf_fin["id_producto"]}',
        f'Huerfanos id_cliente            : {huerf_fin["id_cliente"]}',
        f'Huerfanos id_tienda             : {huerf_fin["id_tienda"]}',
        f'Huerfanos id_promocion          : {huerf_fin["id_promocion"]}',
        '',
        '--- CARGA ---',
        f'Carpeta de salida             : {PROC_DIR}',
        'Tablas cargadas               : dim_cliente, dim_producto, dim_tienda, '
        'dim_promocion, dim_tiempo, fact_ventas',
        f'Tiempo total (seg)            : {tiempo_seg}',
        '=' * 64,
    ]
    with open(ruta, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lineas))
    print(f'Reporte guardado en: {ruta}\n')
    print('\n'.join(lineas))

generar_reporte(
    cal_inicial, cal_final_fact, huerfanos_ini, huerfanos_reporte, dup_negocio_ini,
    tiempo_total, f'{PROC_DIR}/reporte_auditoria_etl.txt',
)


=== CALIDAD FINAL (fact_ventas transformada) ===
  fact_ventas   : filas= 59,700 | nulos=     0 | filas_duplicadas=0
Reporte guardado en: ../data/processed/reporte_auditoria_etl.txt

REPORTE DE AUDITORIA - ETL DATAMART TECHSTORE
Fecha de ejecucion: 2026-07-12 22:39:28

--- FUENTE (fact_ventas) ---
Registros extraidos          : 60,900
Duplicados por llave negocio  : 900
Huerfanos id_producto (inicial): 307

--- TRANSFORMACION ---
Registros tras limpieza       : 59,700
Registros eliminados          : 1,200

--- CALIDAD DE DATOS (fact_ventas) ---
Nulos antes                   : 35,928
Nulos despues                 : 0
Filas duplicadas antes        : 291
Filas duplicadas despues      : 0

--- INTEGRIDAD REFERENCIAL FINAL ---
Huerfanos id_producto          : 0
Huerfanos id_cliente            : 0
Huerfanos id_tienda             : 0
Huerfanos id_promocion          : 0

--- CARGA ---
Carpeta de salida             : ../data/processed
Tablas cargadas               : dim_cliente, dim_producto, d

## 9. Resumen final

Al terminar de ejecutar este notebook, `data/processed/` contendrá:

| Archivo | Contenido |
|---|---|
| `dim_cliente.csv` | Dimensión Cliente limpia |
| `dim_producto.csv` | Dimensión Producto limpia |
| `dim_tienda.csv` | Dimensión Tienda limpia |
| `dim_promocion.csv` | Dimensión Promoción limpia (incluye miembro "Sin Promocion") |
| `dim_tiempo.csv` | Dimensión Tiempo limpia |
| `fact_ventas.csv` | Tabla de hechos con `cantidad`, `precio_unitario`, `descuento`, `importe`, `costo_total`, `margen` |
| `diccionario_datos.csv` | Diccionario de datos (campo, tipo, llave, descripción) |
| `reporte_auditoria_etl.txt` | Reporte de auditoría con métricas antes/después |

Estos CSV quedan listos para importarse a Power BI y construir el modelo estrella
(relaciones desde `fact_ventas` hacia cada dimensión por su llave: `id_cliente`,
`id_producto`, `id_tienda`, `id_promocion`, `fecha`).

In [17]:
# Celda 19 - Vista final de las tablas cargadas
for nombre, df in TABLAS.items():
    print(f'{nombre:<14}: {len(df):>7,} filas | {len(df.columns)} columnas')


fact_ventas   :  59,700 filas | 12 columnas
dim_cliente   :   5,000 filas | 7 columnas
dim_producto  :     500 filas | 7 columnas
dim_tienda    :      15 filas | 5 columnas
dim_promocion :      41 filas | 6 columnas
dim_tiempo    :     730 filas | 7 columnas
